# Hierarchical clustering on the STL-10 embeddings

Agglomerative clustering builds the groups bottom-up: every image starts alone, and the
two closest clusters are merged again and again until only one is left.

Ward linkage merges the pair that increases the within-cluster variance the least. It needs
Euclidean distance; a different metric forces a different linkage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import normalized_mutual_info_score, silhouette_score

DATA = "../week_2/data"
IMG = 96

d = np.load(f"{DATA}/embeddings_stl10_1000.npz")

X = d["E_l12"]
y = d["y"]

class_names = [l.strip() for l in open(f"{DATA}/stl10_binary/class_names.txt") if l.strip()]

def load_rgb(split):
    A = np.fromfile(f"{DATA}/stl10_binary/{split}_X.bin", dtype=np.uint8).reshape(-1, 3, IMG, IMG)
    return A.transpose(0, 3, 2, 1)

X_img = np.concatenate([load_rgb("train"), load_rgb("test")])[d["idx"]]

K = 10
RANDOM_STATE = 0

print(X.shape, X_img.shape, y.shape)

### The dendrogram

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

Z = linkage(X, method="ward")

fig, ax = plt.subplots(figsize=(14, 4))
dendrogram(Z, truncate_mode="level", p=5, ax=ax, no_labels=True)
ax.set_ylabel("distance")
plt.tight_layout()
plt.show()

### Cut the tree at K = 10

The dendrogram shows *all* possible numbers of clusters at once; cutting it at one height
decides how many we get.

In [ ]:
labels = fcluster(Z, t=K, criterion="maxclust") - 1

print("cluster sizes:", np.bincount(labels, minlength=K))

In [ ]:
def report(name, labels):
    print(f"{name}")
    print(f"  NMI        {normalized_mutual_info_score(y, labels):.3f}")
    print(f"  silhouette {silhouette_score(X, labels):.3f}")

### What ended up in each cluster?

In [ ]:
pd.crosstab(pd.Series([class_names[v - 1] for v in y], name="class"),
            pd.Series(labels, name="cluster"))

### Looking at the clusters

In [ ]:
counts = np.bincount(labels, minlength=K)

fig, axes = plt.subplots(K, 11, figsize=(15, K * 1.45),
                         gridspec_kw={"width_ratios": [0.35] + [1] * 10})

for row in range(K):
    axes[row, 0].axis("off")
    axes[row, 0].text(0.5, 0.5, f"cluster {row}\n{counts[row]} imgs",
                      ha="center", va="center", fontsize=8)

    members = np.where(labels == row)[0][:10]

    for col in range(1, 11):
        axes[row, col].axis("off")
        if col - 1 < len(members):
            axes[row, col].imshow(X_img[members[col - 1]])

plt.tight_layout()
plt.show()

### Reading the result

Ward produces the cleanest grouping of the three methods: the contingency table is close to
diagonal, and it is by far the fastest, because it never iterates on an assignment step.

It also gives you something the other two cannot: the whole merge history. You can look at
the dendrogram and ask whether 10 was even the right number to cut at.